In [ ]:
import os
os.environ["PYTHONUTF8"] = "1"

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

In [ ]:
dataset = load_dataset("opus_books", "en-es")
dataset = dataset["train"].train_test_split(test_size=0.05, seed=42)

def rename_columns(example):
    return {
        "src": example["translation"]["en"],
        "tgt": example["translation"]["es"],
    }

dataset = dataset.map(rename_columns, remove_columns=dataset["train"].column_names)

print(dataset)
print(dataset["train"][0])

In [ ]:
def build_prompt(src, tgt=None):
    user = f"Translate this from English to Spanish:\n{src}"
    if tgt is None:
        return f"User: {user}\nAssistant:"
    else:
        return f"User: {user}\nAssistant: {tgt}"

print(build_prompt("Hello"))

In [ ]:
baseline_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

baseline_tokenizer = AutoTokenizer.from_pretrained(baseline_model_id, use_fast=True)
baseline_tokenizer.pad_token = baseline_tokenizer.eos_token

baseline_model = AutoModelForCausalLM.from_pretrained(
    baseline_model_id,
    device_map="auto",
)

In [ ]:
def generate_translation(model, tokenizer, src_text, max_new_tokens=40):
    model.eval()
    prompt = (
        "You are a translation system. "
        "Translate the following text from English to Spanish. "
        "Return only the Spanish translation.\n\n"
        f"English: {src_text}\nSpanish:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "Spanish:" in decoded:
        return decoded.split("Spanish:")[-1].strip()
    return decoded.strip()

In [ ]:
short_text = "Hello, how are you?"
baseline_prediction = generate_translation(baseline_model, baseline_tokenizer, short_text)

print("SOURCE:")
print(short_text)
print("\nBASELINE OUTPUT:")
print(baseline_prediction)

In [ ]:
sample_text = dataset["test"][0]["src"]
reference_text = dataset["test"][0]["tgt"]

baseline_prediction = generate_translation(baseline_model, baseline_tokenizer, sample_text)

print("SOURCE:")
print(sample_text)
print("\nREFERENCE:")
print(reference_text)
print("\nBASELINE OUTPUT:")
print(baseline_prediction)

In [ ]:
test_subset = dataset["test"].select(range(20))

baseline_predictions = []
references = []
sources = []

for example in test_subset:
    src = example["src"]
    tgt = example["tgt"]

    pred = generate_translation(baseline_model, baseline_tokenizer, src)

    sources.append(src)
    references.append(tgt)
    baseline_predictions.append(pred)

print("Done generating baseline predictions.")
print("Example prediction:", baseline_predictions[0])

In [ ]:
bleu = evaluate.load("sacrebleu")

baseline_bleu = bleu.compute(
    predictions=baseline_predictions,
    references=[[ref] for ref in references]
)

print("Baseline BLEU:", baseline_bleu)